# MNIST Smoothing Experiment: Case Alignment and Sensitivity

This notebook compares **non-smoothed** vs **smoothed** explanation maps on MNIST and measures:

- **Case Align robustness** ($S^+$, like-only neighborhood alignment)
- **Captum sensitivity** (`sensitivity_max`)

It also visualizes side-by-side examples with robustness and sensitivity scores.

The experiment is configured to reuse your existing MNIST model and explanation artifact paths in this repository.

In [1]:
# Imports and path setup
import sys
from pathlib import Path
import json

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

from captum.attr import IntegratedGradients, DeepLift
from captum.metrics import sensitivity_max
from scipy.stats import pearsonr, spearmanr
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

current_dir = Path.cwd()
if current_dir.name == "experiments":
    SRC_DIR = current_dir.parent
elif (current_dir / "src").exists():
    SRC_DIR = current_dir / "src"
else:
    SRC_DIR = current_dir

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from case_align.metrics import rankdata, safe_normalise_rows
from explainers.lrp import LRP
from train_mnist_model import MNISTNet, set_seed

sns.set_theme(style="whitegrid")
print(f"Current directory: {current_dir}")
print(f"Using SRC_DIR: {SRC_DIR}")

Current directory: /Users/craigpirie/Documents/PhD/AGREE_ICCBR/agree-iccbr/src/experiments
Using SRC_DIR: /Users/craigpirie/Documents/PhD/AGREE_ICCBR/agree-iccbr/src


In [2]:
# Experiment configuration
MODEL_PATH = SRC_DIR / "models" / "mnist" / "mnist_best_model.pt"
ARTIFACT_PATH = SRC_DIR / "explanations" / "mnist" / "mnist_explanations.pt"
DATA_DIR = SRC_DIR / "data"
OUTPUT_DIR = SRC_DIR / "results" / "mnist_smoothing_case_align_sensitivity"

# None -> use methods stored in artifact
METHODS = None

K = 5
SIM_METRIC = "gower"  # gower | cosine | spearman
PERTURB_RADIUS = 0.1
N_PERTURB_SAMPLES = 10
RETRIEVAL_BATCH_SIZE = 256

SMOOTH_SIGMA = 1.0
SMOOTH_KERNEL_SIZE = 5

MAX_QUERY_SAMPLES = None  # e.g. 64 for faster iteration
SEED = 42

VIS_METHOD = "ig"
N_VIS_EXAMPLES = 6

METHOD_LABELS = {
    "ig": "Integrated Gradients",
    "dl": "DeepLift",
    "lrp": "LRP",
}

print("Configuration ready.")
print(f"Artifact: {ARTIFACT_PATH}")
print(f"Model:    {MODEL_PATH}")

Configuration ready.
Artifact: /Users/craigpirie/Documents/PhD/AGREE_ICCBR/agree-iccbr/src/explanations/mnist/mnist_explanations.pt
Model:    /Users/craigpirie/Documents/PhD/AGREE_ICCBR/agree-iccbr/src/models/mnist/mnist_best_model.pt


## Utility Functions

In [3]:
def _configure_ssl_for_macos() -> None:
    import ssl
    ssl._create_default_https_context = ssl._create_unverified_context


def load_model(model_path: Path, device: torch.device) -> MNISTNet:
    if not model_path.exists():
        raise FileNotFoundError(f"Model not found: {model_path}")

    checkpoint = torch.load(model_path, map_location=device)
    model = MNISTNet()

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
    elif isinstance(checkpoint, dict):
        model.load_state_dict(checkpoint)
    else:
        raise RuntimeError("Unsupported checkpoint format")

    model.to(device)
    model.eval()
    return model


def load_full_test_split(data_dir: Path):
    _configure_ssl_for_macos()

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
    ])

    test_dataset = datasets.MNIST(str(data_dir), train=False, download=True, transform=transform)
    loader = DataLoader(test_dataset, batch_size=512, shuffle=False, num_workers=0)

    all_images = []
    all_labels = []
    for images, labels in loader:
        all_images.append(images)
        all_labels.append(labels)

    images_tensor = torch.cat(all_images, dim=0).float()
    labels_np = torch.cat(all_labels, dim=0).numpy().astype(int)
    return images_tensor, labels_np


def predict_labels_for_images(model, images, batch_size, device):
    pred_chunks = []
    conf_chunks = []

    with torch.no_grad():
        for start in range(0, len(images), batch_size):
            end = min(start + batch_size, len(images))
            xb = images[start:end].to(device)
            logits = model(xb)
            probs = torch.softmax(logits, dim=1)
            conf, preds = probs.max(dim=1)

            pred_chunks.append(preds.detach().cpu().numpy().astype(int))
            conf_chunks.append(conf.detach().cpu().numpy().astype(float))

    return np.concatenate(pred_chunks, axis=0), np.concatenate(conf_chunks, axis=0)


class MNISTLogitsWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        return self.model(x, return_logits=True)


def build_explainers(model, methods):
    explainers = {}
    logits_model = MNISTLogitsWrapper(model)

    for method in methods:
        if method == "ig":
            explainers[method] = IntegratedGradients(model)
        elif method == "dl":
            explainers[method] = DeepLift(logits_model)
        elif method == "lrp":
            explainers[method] = LRP(logits_model)
        else:
            raise ValueError(f"Unknown method: {method}")

    return explainers


def build_metric_context(matrix, sim_metric, epsilon=1e-8):
    matrix_raw = np.asarray(matrix, dtype=float)
    context = {
        "matrix_raw": matrix_raw,
        "epsilon": float(epsilon),
        "sim_metric_name": sim_metric,
    }

    if sim_metric == "gower":
        ranges = np.ptp(matrix_raw, axis=0)
        context["ranges"] = np.where(ranges == 0, 1.0, ranges)
    elif sim_metric == "cosine":
        context["matrix_repr"] = safe_normalise_rows(matrix_raw, eps=epsilon)
    elif sim_metric == "spearman":
        ranked = np.vstack([rankdata(row) for row in matrix_raw])
        ranked = ranked - ranked.mean(axis=1, keepdims=True)
        context["matrix_repr"] = safe_normalise_rows(ranked, eps=epsilon)
    else:
        raise ValueError(f"Unknown sim_metric: {sim_metric}")

    return context


def row_distances(query_vec, context):
    sim_metric = context["sim_metric_name"]
    epsilon = context["epsilon"]
    matrix_raw = context["matrix_raw"]

    if sim_metric == "gower":
        ranges = context["ranges"]
        dist = np.mean(np.abs(matrix_raw - query_vec[None, :]) / ranges[None, :], axis=1)
        return np.clip(dist, 0.0, 1.0)

    if sim_metric == "cosine":
        matrix_repr = context["matrix_repr"]
        query_repr = safe_normalise_rows(np.asarray(query_vec, dtype=float)[None, :], eps=epsilon)[0]
        similarity = matrix_repr @ query_repr
        return np.clip(1.0 - 0.5 * (similarity + 1.0), 0.0, 1.0)

    if sim_metric == "spearman":
        matrix_repr = context["matrix_repr"]
        q_rank = rankdata(np.asarray(query_vec, dtype=float))
        q_rank = q_rank - q_rank.mean()
        query_repr = safe_normalise_rows(q_rank[None, :], eps=epsilon)[0]
        similarity = matrix_repr @ query_repr
        return np.clip(1.0 - 0.5 * (similarity + 1.0), 0.0, 1.0)

    raise ValueError(f"Unknown sim_metric: {sim_metric}")


def compute_case_align_like_only(
    query_index,
    query_label,
    retrieval_labels,
    problem_context,
    solution_context,
    k,
    epsilon=1e-8,
):
    query_problem = problem_context["matrix_raw"][query_index]
    dprob_all = row_distances(query_problem, problem_context)

    like_mask = retrieval_labels == query_label
    like_mask[query_index] = False
    like_indices = np.where(like_mask)[0]

    if like_indices.size == 0:
        return 0.0, 0, False

    like_dists = dprob_all[like_indices]
    order = np.argsort(like_dists)
    k_use = min(k, like_indices.size)
    neigh_indices = like_indices[order[:k_use]]
    dprob_neigh = dprob_all[neigh_indices]

    query_solution = solution_context["matrix_raw"][query_index]
    dsoln_all = row_distances(query_solution, solution_context)
    dsoln_neigh = dsoln_all[neigh_indices]

    ds_min = float(np.min(dsoln_all))
    ds_max = float(np.max(dsoln_all))
    denom = max(ds_max - ds_min, epsilon)
    align = 1.0 - (dsoln_neigh - ds_min) / denom

    weights = 1.0 - dprob_neigh
    weight_sum = float(np.sum(weights))
    if weight_sum <= 0:
        return 0.0, int(k_use), True

    s_plus = float(np.sum(weights * align) / weight_sum)
    return s_plus, int(k_use), True


def gaussian_kernel2d(kernel_size, sigma, device, dtype):
    coords = torch.arange(kernel_size, device=device, dtype=dtype) - (kernel_size - 1) / 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    kernel = torch.outer(g, g)
    kernel = kernel / kernel.sum()
    return kernel.unsqueeze(0).unsqueeze(0)


def smooth_attributions(attrs, sigma=None, kernel_size=5):
    if sigma is None or sigma <= 0:
        return attrs

    if kernel_size % 2 == 0:
        kernel_size += 1

    kernel = gaussian_kernel2d(
        kernel_size=kernel_size,
        sigma=float(sigma),
        device=attrs.device,
        dtype=attrs.dtype,
    )

    channels = attrs.shape[1]
    kernel = kernel.expand(channels, 1, kernel_size, kernel_size)
    return F.conv2d(attrs, kernel, padding=kernel_size // 2, groups=channels)


def compute_retrieval_attributions(
    explainer,
    method,
    images,
    pred_labels,
    baseline,
    batch_size,
    device,
    smooth_sigma=None,
    smooth_kernel_size=5,
):
    n_total = len(images)
    n_batches = (n_total + batch_size - 1) // batch_size
    chunks = []

    for batch_idx, start in enumerate(range(0, n_total, batch_size), start=1):
        end = min(start + batch_size, n_total)
        xb = images[start:end].to(device).clone().detach().requires_grad_(True)
        targets = torch.from_numpy(pred_labels[start:end]).to(device)

        if method in {"ig", "dl"}:
            b = baseline.to(device).expand_as(xb)
            attrs = explainer.attribute(xb, baselines=b, target=targets)
        else:
            attrs = explainer.attribute(xb, target=targets)

        attrs = smooth_attributions(attrs, sigma=smooth_sigma, kernel_size=smooth_kernel_size)
        chunks.append(attrs.detach().cpu())

        if batch_idx == 1 or batch_idx == n_batches or batch_idx % max(n_batches // 10, 1) == 0:
            print(f"      attribution batches: {batch_idx}/{n_batches}")

    return torch.cat(chunks, dim=0).float()


def compute_sensitivity(
    explainer,
    method,
    image,
    target,
    baseline,
    perturb_radius,
    n_perturb_samples,
    device,
    smooth_sigma=None,
    smooth_kernel_size=5,
):
    x = image.unsqueeze(0).to(device)

    if method in {"ig", "dl"}:
        b0 = baseline.to(device)

        def explain_func(inputs, target=None):
            x_in = inputs[0] if isinstance(inputs, tuple) else inputs
            b = b0.expand_as(x_in)
            attrs = explainer.attribute(x_in, baselines=b, target=target)
            return smooth_attributions(attrs, sigma=smooth_sigma, kernel_size=smooth_kernel_size)

    else:
        def explain_func(inputs, target=None):
            x_in = inputs[0] if isinstance(inputs, tuple) else inputs
            attrs = explainer.attribute(x_in, target=target)
            return smooth_attributions(attrs, sigma=smooth_sigma, kernel_size=smooth_kernel_size)

    sens = sensitivity_max(
        explanation_func=explain_func,
        inputs=x,
        target=target,
        perturb_radius=perturb_radius,
        n_perturb_samples=n_perturb_samples,
    )
    return float(sens.detach().cpu().item())


def evaluate_variant(
    method,
    variant_name,
    explainer,
    query_images,
    query_labels,
    query_pred_labels,
    query_confidences,
    query_retrieval_indices,
    retrieval_images,
    retrieval_labels,
    retrieval_pred_labels,
    baseline,
    k,
    sim_metric,
    perturb_radius,
    n_perturb_samples,
    retrieval_batch_size,
    device,
    smooth_sigma=None,
    smooth_kernel_size=5,
):
    n_queries = query_images.shape[0]
    n_retrieval = retrieval_images.shape[0]

    print(f"  [{method}/{variant_name}] building retrieval attributions ({n_retrieval} samples)...")
    retrieval_attributions = compute_retrieval_attributions(
        explainer=explainer,
        method=method,
        images=retrieval_images,
        pred_labels=retrieval_pred_labels,
        baseline=baseline,
        batch_size=retrieval_batch_size,
        device=device,
        smooth_sigma=smooth_sigma,
        smooth_kernel_size=smooth_kernel_size,
    )

    X_flat = retrieval_images.view(n_retrieval, -1).detach().cpu().numpy()
    E_flat = retrieval_attributions.view(n_retrieval, -1).detach().cpu().numpy()

    problem_context = build_metric_context(X_flat, sim_metric=sim_metric)
    solution_context = build_metric_context(E_flat, sim_metric=sim_metric)

    rows = []
    for i in range(n_queries):
        retrieval_index = int(query_retrieval_indices[i])

        s_plus, like_count, has_like_neighbour = compute_case_align_like_only(
            query_index=retrieval_index,
            query_label=int(query_labels[i]),
            retrieval_labels=retrieval_labels,
            problem_context=problem_context,
            solution_context=solution_context,
            k=k,
        )

        sensitivity = compute_sensitivity(
            explainer=explainer,
            method=method,
            image=query_images[i],
            target=int(query_pred_labels[i]),
            baseline=baseline,
            perturb_radius=perturb_radius,
            n_perturb_samples=n_perturb_samples,
            device=device,
            smooth_sigma=smooth_sigma,
            smooth_kernel_size=smooth_kernel_size,
        )

        rows.append({
            "method": method,
            "variant": variant_name,
            "sample_position": int(i),
            "original_test_index": retrieval_index,
            "true_label": int(query_labels[i]),
            "pred_label": int(query_pred_labels[i]),
            "confidence": float(query_confidences[i]),
            "case_align_like_count": int(like_count),
            "case_align_has_like_neighbour": bool(has_like_neighbour),
            "case_align_S_plus": float(s_plus) if has_like_neighbour else np.nan,
            "captum_sensitivity": float(sensitivity),
        })

        if (i + 1) % max(n_queries // 10, 1) == 0 or i == n_queries - 1:
            print(f"      evaluated samples: {i + 1}/{n_queries}")

    query_idx_tensor = torch.from_numpy(query_retrieval_indices.astype(np.int64))
    query_attributions = retrieval_attributions.index_select(0, query_idx_tensor).detach().cpu()

    return pd.DataFrame(rows), query_attributions

## Load Artifacts, Model, and Retrieval Pool

In [4]:
set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if not ARTIFACT_PATH.exists():
    raise FileNotFoundError(f"Explanation artifact not found: {ARTIFACT_PATH}")
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model checkpoint not found: {MODEL_PATH}")

artifact = torch.load(ARTIFACT_PATH, map_location="cpu")
artifact_methods = [m.lower() for m in artifact.get("methods", ["ig", "dl", "lrp"])]
methods_to_run = artifact_methods if METHODS is None else [m.lower() for m in METHODS]
methods_to_run = [m for m in methods_to_run if m in {"ig", "dl", "lrp"}]

if not methods_to_run:
    raise RuntimeError("No valid methods selected for evaluation.")

query_images = artifact["images"].float()
query_labels = artifact["labels"].numpy().astype(int)
query_pred_labels = artifact["pred_labels"].numpy().astype(int)
query_confidences = artifact["confidences"].numpy().astype(float)
query_retrieval_indices = artifact["sample_indices"].numpy().astype(int)
baseline = artifact["baseline"].float()

if MAX_QUERY_SAMPLES is not None and len(query_images) > MAX_QUERY_SAMPLES:
    query_images = query_images[:MAX_QUERY_SAMPLES]
    query_labels = query_labels[:MAX_QUERY_SAMPLES]
    query_pred_labels = query_pred_labels[:MAX_QUERY_SAMPLES]
    query_confidences = query_confidences[:MAX_QUERY_SAMPLES]
    query_retrieval_indices = query_retrieval_indices[:MAX_QUERY_SAMPLES]

model = load_model(MODEL_PATH, device)
explainers = build_explainers(model, methods_to_run)

print("Loading full MNIST test split for retrieval...")
retrieval_images, retrieval_labels = load_full_test_split(DATA_DIR)
retrieval_pred_labels, retrieval_confidences = predict_labels_for_images(
    model=model,
    images=retrieval_images,
    batch_size=RETRIEVAL_BATCH_SIZE,
    device=device,
)

if np.any(query_retrieval_indices < 0) or np.any(query_retrieval_indices >= len(retrieval_labels)):
    raise IndexError("Some query sample_indices fall outside retrieval pool range.")

print(f"Query samples: {len(query_images)}")
print(f"Retrieval pool size: {len(retrieval_images)}")
print(f"Methods: {methods_to_run}")

Using device: cpu
Loading full MNIST test split for retrieval...
Query samples: 24
Retrieval pool size: 10000
Methods: ['ig', 'dl', 'lrp']


## Evaluate Non-Smooth vs Smooth Explanations

In [ ]:
scores_parts = []
query_attr_store = {}

for method in methods_to_run:
    print(f"\n=== Evaluating {METHOD_LABELS.get(method, method.upper())} ===")

    raw_df, raw_query_attr = evaluate_variant(
        method=method,
        variant_name="raw",
        explainer=explainers[method],
        query_images=query_images,
        query_labels=query_labels,
        query_pred_labels=query_pred_labels,
        query_confidences=query_confidences,
        query_retrieval_indices=query_retrieval_indices,
        retrieval_images=retrieval_images,
        retrieval_labels=retrieval_labels,
        retrieval_pred_labels=retrieval_pred_labels,
        baseline=baseline,
        k=K,
        sim_metric=SIM_METRIC,
        perturb_radius=PERTURB_RADIUS,
        n_perturb_samples=N_PERTURB_SAMPLES,
        retrieval_batch_size=RETRIEVAL_BATCH_SIZE,
        device=device,
        smooth_sigma=None,
        smooth_kernel_size=SMOOTH_KERNEL_SIZE,
    )

    smooth_df, smooth_query_attr = evaluate_variant(
        method=method,
        variant_name="smooth",
        explainer=explainers[method],
        query_images=query_images,
        query_labels=query_labels,
        query_pred_labels=query_pred_labels,
        query_confidences=query_confidences,
        query_retrieval_indices=query_retrieval_indices,
        retrieval_images=retrieval_images,
        retrieval_labels=retrieval_labels,
        retrieval_pred_labels=retrieval_pred_labels,
        baseline=baseline,
        k=K,
        sim_metric=SIM_METRIC,
        perturb_radius=PERTURB_RADIUS,
        n_perturb_samples=N_PERTURB_SAMPLES,
        retrieval_batch_size=RETRIEVAL_BATCH_SIZE,
        device=device,
        smooth_sigma=SMOOTH_SIGMA,
        smooth_kernel_size=SMOOTH_KERNEL_SIZE,
    )

    scores_parts.extend([raw_df, smooth_df])
    query_attr_store[(method, "raw")] = raw_query_attr
    query_attr_store[(method, "smooth")] = smooth_query_attr

scores_df = pd.concat(scores_parts, ignore_index=True)
scores_df["method_label"] = scores_df["method"].map(lambda m: METHOD_LABELS.get(m, m.upper()))

print("\nEvaluation complete.")
print(scores_df.head())

In [ ]:
# Summary tables and correlations
summary_df = (
    scores_df
    .groupby(["method", "variant"], as_index=False)
    .agg(
        n_samples=("sample_position", "count"),
        mean_case_align_S_plus=("case_align_S_plus", "mean"),
        std_case_align_S_plus=("case_align_S_plus", "std"),
        mean_captum_sensitivity=("captum_sensitivity", "mean"),
        std_captum_sensitivity=("captum_sensitivity", "std"),
    )
)
summary_df["method_label"] = summary_df["method"].map(lambda m: METHOD_LABELS.get(m, m.upper()))

corr_rows = []
for (method, variant), group in scores_df.groupby(["method", "variant"]):
    valid = group.dropna(subset=["case_align_S_plus", "captum_sensitivity"])
    if (
        len(valid) >= 2
        and valid["case_align_S_plus"].nunique() > 1
        and valid["captum_sensitivity"].nunique() > 1
    ):
        p_r, p_p = pearsonr(valid["case_align_S_plus"], valid["captum_sensitivity"])
        s_r, s_p = spearmanr(valid["case_align_S_plus"], valid["captum_sensitivity"])
    else:
        p_r = p_p = s_r = s_p = np.nan

    corr_rows.append({
        "method": method,
        "variant": variant,
        "pearson_case_align_vs_sensitivity": p_r,
        "pearson_p_value": p_p,
        "spearman_case_align_vs_sensitivity": s_r,
        "spearman_p_value": s_p,
    })

corr_df = pd.DataFrame(corr_rows)
corr_df["method_label"] = corr_df["method"].map(lambda m: METHOD_LABELS.get(m, m.upper()))

delta_df = scores_df.pivot_table(
    index=["method", "sample_position"],
    columns="variant",
    values=["case_align_S_plus", "captum_sensitivity"],
)
delta_df.columns = [f"{metric}_{variant}" for metric, variant in delta_df.columns]
delta_df = delta_df.reset_index()
delta_df["delta_case_align"] = delta_df["case_align_S_plus_smooth"] - delta_df["case_align_S_plus_raw"]
delta_df["delta_sensitivity"] = delta_df["captum_sensitivity_smooth"] - delta_df["captum_sensitivity_raw"]

print("Method/variant summary:")
print(summary_df[[
    "method_label", "variant", "n_samples",
    "mean_case_align_S_plus", "std_case_align_S_plus",
    "mean_captum_sensitivity", "std_captum_sensitivity",
]].round(4).to_string(index=False))

print("\nCorrelation (Case Align vs Sensitivity):")
print(corr_df[[
    "method_label", "variant",
    "pearson_case_align_vs_sensitivity", "pearson_p_value",
    "spearman_case_align_vs_sensitivity", "spearman_p_value",
]].round(4).to_string(index=False))

print("\nAverage smooth - raw deltas by method:")
print(delta_df.groupby("method")[["delta_case_align", "delta_sensitivity"]].mean().round(4))

## Quantitative Visualizations

In [ ]:
method_order = methods_to_run

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(
    data=scores_df,
    x="method",
    y="case_align_S_plus",
    hue="variant",
    order=method_order,
    ax=axes[0],
)
axes[0].set_title("Case Align Robustness ($S^+$): smooth vs non-smooth")
axes[0].set_xlabel("Explainer")
axes[0].set_ylabel("Case Align $S^+$")

sns.boxplot(
    data=scores_df,
    x="method",
    y="captum_sensitivity",
    hue="variant",
    order=method_order,
    ax=axes[1],
)
axes[1].set_title("Captum sensitivity_max: smooth vs non-smooth")
axes[1].set_xlabel("Explainer")
axes[1].set_ylabel("Sensitivity")

for ax in axes:
    ax.set_xticklabels([METHOD_LABELS.get(t.get_text(), t.get_text()) for t in ax.get_xticklabels()])

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.scatterplot(
    data=scores_df,
    x="case_align_S_plus",
    y="captum_sensitivity",
    hue="variant",
    style="method",
    alpha=0.8,
    ax=axes[0],
)
axes[0].set_title("Case Align vs Sensitivity")
axes[0].set_xlabel("Case Align $S^+$")
axes[0].set_ylabel("Sensitivity")

delta_long = delta_df.melt(
    id_vars=["method", "sample_position"],
    value_vars=["delta_case_align", "delta_sensitivity"],
    var_name="delta_metric",
    value_name="delta_value",
)

sns.boxplot(
    data=delta_long,
    x="delta_metric",
    y="delta_value",
    hue="method",
    ax=axes[1],
)
axes[1].set_title("Smooth - raw delta by metric")
axes[1].set_xlabel("Metric")
axes[1].set_ylabel("Delta")
axes[1].set_xticklabels(["Δ Case Align", "Δ Sensitivity"])

plt.tight_layout()
plt.show()

## Qualitative Examples: Smooth vs Non-Smooth Attribution Maps

In [ ]:
if VIS_METHOD not in methods_to_run:
    raise ValueError(f"VIS_METHOD={VIS_METHOD} not in selected methods {methods_to_run}")

method_scores = scores_df[scores_df["method"] == VIS_METHOD].copy()
method_pivot = method_scores.pivot(
    index="sample_position",
    columns="variant",
    values=["case_align_S_plus", "captum_sensitivity"],
)
method_pivot.columns = [f"{metric}_{variant}" for metric, variant in method_pivot.columns]

required_cols = [
    "case_align_S_plus_raw",
    "case_align_S_plus_smooth",
    "captum_sensitivity_raw",
    "captum_sensitivity_smooth",
]
missing_cols = [c for c in required_cols if c not in method_pivot.columns]
if missing_cols:
    raise RuntimeError(f"Missing expected columns for visualization: {missing_cols}")

method_pivot["abs_delta_robustness"] = (
    method_pivot["case_align_S_plus_smooth"] - method_pivot["case_align_S_plus_raw"]
).abs()

selected_positions = method_pivot.sort_values("abs_delta_robustness", ascending=False).head(N_VIS_EXAMPLES).index.tolist()

selected_table = method_pivot.loc[selected_positions, [
    "case_align_S_plus_raw",
    "case_align_S_plus_smooth",
    "captum_sensitivity_raw",
    "captum_sensitivity_smooth",
    "abs_delta_robustness",
]].round(4)

print(f"Selected top-{len(selected_positions)} examples by |Δ robustness| for {METHOD_LABELS.get(VIS_METHOD, VIS_METHOD.upper())}:")
print(selected_table)

raw_attrs = query_attr_store[(VIS_METHOD, "raw")].numpy()
smooth_attrs = query_attr_store[(VIS_METHOD, "smooth")].numpy()

n_rows = len(selected_positions)
fig, axes = plt.subplots(n_rows, 3, figsize=(12, 3.4 * n_rows))
if n_rows == 1:
    axes = np.expand_dims(axes, axis=0)

for row_idx, sample_pos in enumerate(selected_positions):
    ax_img, ax_raw, ax_smooth = axes[row_idx]

    image = query_images[sample_pos, 0].numpy()
    raw_attr = raw_attrs[sample_pos, 0]
    smooth_attr = smooth_attrs[sample_pos, 0]
    vmax = max(np.max(np.abs(raw_attr)), np.max(np.abs(smooth_attr)), 1e-8)

    test_index = int(query_retrieval_indices[sample_pos])
    true_label = int(query_labels[sample_pos])
    pred_label = int(query_pred_labels[sample_pos])

    ax_img.imshow(image, cmap="gray")
    ax_img.set_title(f"Sample {sample_pos} (test idx {test_index})\ntrue={true_label}, pred={pred_label}")
    ax_img.axis("off")

    ax_raw.imshow(raw_attr, cmap="seismic", vmin=-vmax, vmax=vmax)
    ax_raw.set_title(
        f"Non-smooth\nS+={method_pivot.loc[sample_pos, 'case_align_S_plus_raw']:.3f} | "
        f"Sens={method_pivot.loc[sample_pos, 'captum_sensitivity_raw']:.3f}"
    )
    ax_raw.axis("off")

    ax_smooth.imshow(smooth_attr, cmap="seismic", vmin=-vmax, vmax=vmax)
    ax_smooth.set_title(
        f"Smoothed (σ={SMOOTH_SIGMA})\nS+={method_pivot.loc[sample_pos, 'case_align_S_plus_smooth']:.3f} | "
        f"Sens={method_pivot.loc[sample_pos, 'captum_sensitivity_smooth']:.3f}"
    )
    ax_smooth.axis("off")

fig.suptitle(
    f"{METHOD_LABELS.get(VIS_METHOD, VIS_METHOD.upper())}: smooth vs non-smooth attribution maps with robustness scores",
    y=1.01,
)
plt.tight_layout()
plt.show()

In [ ]:
# Save outputs
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

scores_path = OUTPUT_DIR / "mnist_smooth_vs_raw_scores.csv"
summary_path = OUTPUT_DIR / "mnist_smooth_vs_raw_summary.csv"
corr_path = OUTPUT_DIR / "mnist_smooth_vs_raw_correlations.csv"
delta_path = OUTPUT_DIR / "mnist_smooth_vs_raw_deltas.csv"
config_path = OUTPUT_DIR / "mnist_smooth_vs_raw_config.json"

scores_df.to_csv(scores_path, index=False)
summary_df.to_csv(summary_path, index=False)
corr_df.to_csv(corr_path, index=False)
delta_df.to_csv(delta_path, index=False)

with config_path.open("w", encoding="utf-8") as f:
    json.dump({
        "model_path": str(MODEL_PATH),
        "artifact_path": str(ARTIFACT_PATH),
        "data_dir": str(DATA_DIR),
        "methods": methods_to_run,
        "k": int(K),
        "sim_metric": SIM_METRIC,
        "perturb_radius": float(PERTURB_RADIUS),
        "n_perturb_samples": int(N_PERTURB_SAMPLES),
        "retrieval_batch_size": int(RETRIEVAL_BATCH_SIZE),
        "smooth_sigma": float(SMOOTH_SIGMA),
        "smooth_kernel_size": int(SMOOTH_KERNEL_SIZE),
        "n_query_samples": int(len(query_images)),
        "retrieval_pool_size": int(len(retrieval_images)),
    }, f, indent=2)

print("Saved experiment outputs:")
print(f"- Scores:      {scores_path}")
print(f"- Summary:     {summary_path}")
print(f"- Correlation: {corr_path}")
print(f"- Deltas:      {delta_path}")
print(f"- Config:      {config_path}")